# logging

对应 `stdlib.md`：分级日志。

笔记本在 `python_base/logging/qa.ipynb`。需要读写文件时，工作空间就是这个目录，题目文件落在旁边的子目录里。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

提示：笔记本里反复跑时，根 logger 可能已有 handler；每题前置代码会先清掉旧 handler，避免串台。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_base/logging。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "logging" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_base" / "logging"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_base/logging')

## 1. 基本配置并写一条 info

把日志写到 `box/app.log`，级别设为 `INFO`。写一条内容为 `hello` 的 info。然后打印日志文件全文。


In [2]:
import logging

box = ROOT / "q1"
box.mkdir(parents=True, exist_ok=True)
path = box / "app.log"
path.write_text("", encoding="utf-8")
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

# 作答

logging.root.handlers.append(logging.FileHandler(path))
logging.root.setLevel(logging.INFO)
logging.root.info("hello")

print(path.read_text(encoding="utf-8"))
#评阅
# 对。FileHandler + 级别 INFO + info 写入，文件里有 hello。
# 更常见写法是 logging.basicConfig(filename=..., level=..., force=True)。

#参考答案
# logging.basicConfig(filename=path, level=logging.INFO, force=True)
# logging.info("hello")
# print(path.read_text(encoding="utf-8"))


hello



## 2. 级别过滤

把日志写到 `box/app.log`，级别设为 `WARNING`。先写一条 `debug`（内容 `dbg`），再写一条 `warning`（内容 `warn`）。打印日志文件全文（应只有 warning 那条）。


In [3]:
import logging

box = ROOT / "q2"
box.mkdir(parents=True, exist_ok=True)
path = box / "app.log"
path.write_text("", encoding="utf-8")
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

# 作答

logging.root.handlers.append(logging.FileHandler(path))
logging.root.setLevel(logging.WARNING)

logging.root.debug("dbg")
logging.root.warning("warn")

print(path.read_text(encoding="utf-8"))
#评阅
# 对。级别 WARNING 时 debug 被滤掉，只留下 warn。

#参考答案
# logging.basicConfig(filename=path, level=logging.WARNING, force=True)
# logging.debug("dbg")
# logging.warning("warn")
# print(path.read_text(encoding="utf-8"))


warn



## 3. 具名 logger

拿到名字为 `demo` 的 logger，级别设为 `INFO`，给它加一个写入 `box/app.log` 的 FileHandler（不要依赖 root 的 basicConfig）。用这个 logger 写一条 info，内容 `named`。打印日志文件全文。


In [4]:
import logging

box = ROOT / "q3"
box.mkdir(parents=True, exist_ok=True)
path = box / "app.log"
path.write_text("", encoding="utf-8")

# 作答

logger = logging.getLogger("demo")
logger.setLevel(logging.INFO)
logger.handlers.append(logging.FileHandler(path))

logger.info("named")

print(path.read_text(encoding="utf-8"))
#评阅
# 对。getLogger("demo") + 自己的 FileHandler，不靠 basicConfig。
# 重跑前最好先清空 logger.handlers，或设 propagate=False，避免重复挂 handler。

#参考答案
# logger = logging.getLogger("demo")
# logger.handlers.clear()
# logger.setLevel(logging.INFO)
# logger.propagate = False
# logger.addHandler(logging.FileHandler(path, encoding="utf-8"))
# logger.info("named")
# print(path.read_text(encoding="utf-8"))


named



## 4. 格式化

配置日志写到 `box/app.log`，级别 `INFO`。格式里至少包含级别名和消息，形如 `INFO - hello`（中间用 ` - ` 连接即可）。写一条 info，消息为 `hello`。打印日志文件全文。


In [5]:
import logging

box = ROOT / "q4"
box.mkdir(parents=True, exist_ok=True)
path = box / "app.log"
path.write_text("", encoding="utf-8")
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

# 作答

handler = logging.FileHandler(path, encoding="utf-8")
handler.formatter = logging.Formatter("%(levelname)s - %(message)s")
logging.root.handlers.append(handler)
logging.root.setLevel(logging.INFO)

logging.root.info("hello")

print(path.read_text(encoding="utf-8"))
#评阅
# 对。Formatter("%(levelname)s - %(message)s") 得到 INFO - hello。
# 惯用法是 handler.setFormatter(...)，直接赋 formatter 属性也能用。

#参考答案
# logging.basicConfig(
#     filename=path,
#     level=logging.INFO,
#     format="%(levelname)s - %(message)s",
#     force=True,
# )
# logging.info("hello")
# print(path.read_text(encoding="utf-8"))


INFO - hello



## 5. 记录异常

配置日志写到 `box/app.log`，级别 `ERROR`。在 `try/except` 里制造一个 `ZeroDivisionError`，在 except 里用 logger 记录这条异常（要带上堆栈信息）。打印日志文件全文（应能看到 traceback）。


In [6]:
import logging

box = ROOT / "q5"
box.mkdir(parents=True, exist_ok=True)
path = box / "app.log"
path.write_text("", encoding="utf-8")
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

# 作答

logging.root.handlers.append(logging.FileHandler(path))
logging.root.setLevel(logging.ERROR)

try:
    raise ZeroDivisionError("zero")
except ZeroDivisionError as e:
    logging.root.exception("error")

print(path.read_text(encoding="utf-8"))
#评阅
# 对。exception(...) 会带上 traceback。

#参考答案
# logging.basicConfig(filename=path, level=logging.ERROR, force=True)
# try:
#     1 / 0
# except ZeroDivisionError:
#     logging.exception("error")
# print(path.read_text(encoding="utf-8"))


error
Traceback (most recent call last):
  File "/var/folders/6g/9chj8ddx1033kwmzkbfd95l40000gn/T/ipykernel_62303/2023887270.py", line 16, in <module>
    raise ZeroDivisionError("zero")
ZeroDivisionError: zero



## 6. 子 logger 名字

拿到名字为 `app.db` 的 logger，写一条 info，消息为 `query`，输出到 `box/app.log`（级别 `INFO`，可用 basicConfig 指向该文件）。打印日志文件全文；全文里应能看到 logger 名字 `app.db`（格式里带上 `%(name)s`）。


In [7]:
import logging

box = ROOT / "q6"
box.mkdir(parents=True, exist_ok=True)
path = box / "app.log"
path.write_text("", encoding="utf-8")
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

# 作答

logging.basicConfig(
    filename = path,
    level = logging.INFO,
    format = "%(name)s %(levelname)s %(message)s",
    force = True,
)

logging.getLogger("app.db").info("query")

print(path.read_text(encoding="utf-8"))

#评阅
# 未作答。需要：basicConfig 带 %(name)s；getLogger("app.db").info("query")；打印文件且能看到 app.db。

#参考答案
# logging.basicConfig(
#     filename=path,
#     level=logging.INFO,
#     format="%(name)s %(levelname)s %(message)s",
#     force=True,
# )
# logging.getLogger("app.db").info("query")
# print(path.read_text(encoding="utf-8"))


app.db INFO query

